In [ ]:
# !pip install ultralytics==8.3.227 scikit-learn matplotlib "pandas<3.0.0" livelossplot plotly

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard Library imports
import sys
sys.path.insert(0, "..")
from pathlib import Path

# External imports
import torch
from sklearn.metrics import classification_report
from ultralytics import YOLO
from ultralytics.models.yolo.classify.val import ClassificationValidator

# Local imports
from yolo_callbacks import LossPlotCallbacks
from yolo_dataset import RGBClassificationTrainer
from yolo_tools import find_best_threshold, get_targets_and_confs

In [ ]:
DATA_PATH = Path(r"FILL_ME")
MODEL_NAME = "yolo11n-cls.pt"

# Hyper-parameters
EPOCHS = 100
WORKERS = 4
BATCH = 16
IMGSZ = 224
DROPOUT = 0.2

# Constants
CORRECT_CLASS = "correct"
INCORRECT_CLASS = "incorrect"
RUN_NAME = "run"
PROJECT_NAME = "automatic_mask_cleaner_results"
BG_MODE = "overlay"  # "gray" or "overlay"
RGBClassificationTrainer.bg_mode = BG_MODE
TRAINER = RGBClassificationTrainer
FRACTION = 1.0  # Fraction of the dataset to use for training. Useful for experiments.

In [ ]:
model = YOLO(MODEL_NAME)
run_name_train = f"{RUN_NAME}_train"

cbs = LossPlotCallbacks(
    figpath=f"{PROJECT_NAME}/{run_name_train}/loss_plot.png",
    mode="classification",
    names=["correct", "incorrect"],
)

model.add_callback("on_train_epoch_end", cbs.on_train_epoch_end)
model.add_callback("on_val_batch_end", cbs.on_val_batch_end)
model.add_callback("on_val_end", cbs.on_val_end)

results = model.train(
    data=DATA_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    workers=WORKERS,
    batch=BATCH,
    deterministic=True,
    trainer=TRAINER,
    project=PROJECT_NAME,
    name=run_name_train,
    fraction=FRACTION,
    dropout=DROPOUT,
)

In [ ]:
val_dataloader = ClassificationValidator().get_dataloader(
    Path(DATA_PATH) / "val", BATCH
)

#### Calculate the best conf value on the validation set for the "incorrect" class

In [ ]:
targets, positive_confs = get_targets_and_confs(
    model, val_dataloader, positive_class_name=INCORRECT_CLASS
)
best_t, best_f1 = find_best_threshold(positive_confs, targets, show=True)
print(f"Best threshold: {best_t:.3f}  →  F1: {best_f1:.3f}")

#### Calculate classification score on validation set

In [ ]:
preds = (positive_confs >= best_t).to(torch.uint8)
print(
    classification_report(targets, preds, target_names=model.names.values(), digits=3)
)

#### Calculate classification score on test set

In [ ]:
test_dataloader = ClassificationValidator().get_dataloader(
    Path(DATA_PATH) / "test", BATCH
)
targets_test, positive_confs_test = get_targets_and_confs(
    model, test_dataloader, positive_class_name=INCORRECT_CLASS
)
preds_test = (positive_confs_test >= best_t).to(torch.uint8)
print(
    classification_report(
        targets_test, preds_test, target_names=model.names.values(), digits=3
    )
)